# Chunked-Prefill


[SARATHI: Efficient LLM Inference by Piggybacking Decodes with Chunked Prefills](https://arxiv.org/pdf/2308.16369)

摘选 Abstration

1. Decoding 计算量不充分，memory-bound


> Large Language Model (LLM) inference consists of two distinct phases – prefill phase which processes the input prompt and decode phase which generates output tokens autoregressively. While the prefill phase effectively saturates GPU compute at small batch sizes, the decode phase results in low compute utilization as it generates one token at a time per request. The varying prefill and decode times also lead to imbalance across micro-batches when using pipeline parallelism, resulting in further inefficiency due to bubbles.


2. Decoding 投影搭 Prefill 便车

> We present SARATHI to address these challenges. SARATHI employs chunked-prefills, which splits a **prefill request into equal sized chunks, and decode-maximal batching**, which constructs a batch using a single prefill chunk and populates the **remaining slots with decodes**. During inference, the prefill chunk saturates GPU compute, while the decode requests ‘piggyback’ and cost up to an order of magnitude less compared to a decode-only batch. Chunked-prefills allows constructing multiple decode-maximal batches from a single prefill request, maximizing coverage of decodes that can piggyback. Furthermore, the uniform compute design of these batches ameliorates the imbalance between micro-batches, significantly reducing pipeline bubbles.

这一段还描述了一个特殊的 Feature

"prefill request into equal sized chunks, and decode-maximal batching", 

1. 所有 decoding 请求压成一个 batching,
2. 单一 Prefill 请求可以进行切片，切片数据称为一个 batch，切片是在 input-ids 序列规模上的。


## Part1: Proj 投影搭便车

解码过程特性：

1. Prefill: Compute-bound
2. Decoding: Memory-bound，我们熟知的计算 attention 前要拉取大量的 KVCache，存在显著的 memory 访问的开销。

另外从投影分析：

在 Decoding 时，输入为 next_token, 此时需要拉取一个Attn里的 Wq、Wk、Wv 做投影，或者 FFN 里的 W 权重，计算形式为

`x(1xd) @ Wq(dxd)`

可见我们频繁拉取大的权重矩阵到 SRAM 中，只是做简单的运算，这种访存开销是不经济的。而 Prefill 则是产生了充分的计算的，如：

```
X(Lxd) @ Wq(dxd)
```

此时如果我们定义两个请求：其输入为 
```
req1 (prefill stage): X_req1 [1000xd]
req2 (decoding stage): x_req2 [1xd]
```

我们进行拼接为
```
X_cp = torch.cat( [X_req1, x_req2], dim=0)
Q_cp = X_cp @ Wq
Q_req1, q_req2 <- split(Q_cp)
```

这种处理技巧我们称为 Chunked—Prefill。 为什么不叫 Chunked-Decoding?

1. Prefill 在投影计算中是矩阵乘高效的
2. Decoding 搭了 Prefill 的便车
3. Chunked 的定义，我理解是 `X_cp` 对应有多块（chunked）数据来源

可以理解为通过融合 PD 阶段共性计算，减少 memory-visited 开销，从而提速。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from typing import Dict, List, Set, Tuple, Optional, Any, Deque

torch.manual_seed(42)

In [ ]:
d = 2048

# Prefill 和 Decoding 各一个请求
L_p = 1024
L_d = 1

X_p = torch.randn(L_p, d)
X_d = torch.randn(L_d, d)

W = torch.randn(d, d)

# combine
X_pd = torch.cat((X_p, X_d), dim=0)

Y_pd = X_pd @ W

Yp = Y_pd[:L_p, :]
Yd = Y_pd[L_p:, :]

In [ ]:
Wq = nn.Linear(d, d)


def ChunkPrefillLinearForward(W, XP, XD):
    with torch.no_grad():
        L_p, d = XP.shape
        XPD = torch.cat((XP, XD), dim=0)
        YPD = W(XPD)
        return YPD[:L_p, :], YPD[L_p:, :]


Yp, Yd = ChunkPrefillLinearForward(Wq, X_p, X_d)
print(Yp.shape)
print(Yd.shape)

torch.Size([1024, 2048])
torch.Size([1, 2048])


In [ ]:
# 通用实现

# config
d = 2048
bsz_p = 3  # prefill请求数量
bsz_d = 20  # decoding请求数量
seq_p = 1024
seq_d = 1

# data
Wq = nn.Linear(d, d)


def ChunkPrefillLinearForward(W, XP, XD):
    """
    更通用的 Chunk Prefill, 可处理 PD batch size 不同的情况
    """

    BP, LP, D = XP.shape
    BD, LD, D = XD.shape

    with torch.no_grad():
        XP = XP.reshape(BP*LP, D)
        XD = XD.reshape(BD*LD, D)
        XPD = torch.cat((XP, XD), dim=0)
        YPD = W(XPD)

        YP = YPD[:BP*LP].reshape(BP, LP, D)
        YD = YPD[BP*LP:].reshape(BD, LD, D)

        return YP, YD


X_p = torch.randn(bsz_p, seq_p, d)
X_d = torch.randn(bsz_d, seq_d, d)

Yp, Yd = ChunkPrefillLinearForward(Wq, X_p, X_d)
print(Yp.shape)
print(Yd.shape)

torch.Size([3, 1024, 2048])
torch.Size([20, 1, 2048])


## Part2. Chunked-Prefill


当一个请求长度为 20, page 长度为 8, 那么可以切成 3 个 page。 

1. 前向计算逻辑？
2. 此计算与 context-parallelsim 的区别是什么？


### Basic Prefill 实现

In [ ]:
bsz = 1
seq_len = 20
page_size = 8
dim = 16
vocab_size = 100

In [ ]:
# Basic Model

def attention_kernel(Q, K, V):
    S = Q @ K.transpose(1, 2)
    P = F.softmax(S, dim=-1)
    Z = P @ V
    return Z


class Attention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.Wq = nn.Linear(dim, dim)
        self.Wk = nn.Linear(dim, dim)
        self.Wv = nn.Linear(dim, dim)
        self.Wo = nn.Linear(dim, dim)

    def forward(self, X, KVCache=None):
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)
        Z = attention_kernel(Q, K, V)
        O = self.Wo(Z)
        return O, [K, V]


class XDGModel(nn.Module):
    def __init__(self, dim, vocab_size, attention_layer):
        super().__init__()
        self.embd = nn.Embedding(vocab_size, dim)
        self.decoder = attention_layer(dim)
        self.lm_head = nn.Linear(dim, vocab_size)

    def forward(self, x, KVCache=None):
        X = self.embd(x)
        H, KV = self.decoder(X, KVCache=KVCache)
        logits = self.lm_head(H)
        return logits, KV

In [7]:
model = XDGModel(dim, vocab_size, attention_layer=Attention)
# X = torch.randn(bsz, seq_len, dim)
x = torch.randint(vocab_size, (bsz, seq_len))
logits, KVCache = model(x)
print(x.shape)
print(logits.shape)
print(KVCache[0].shape)

torch.Size([1, 20])
torch.Size([1, 20, 100])
torch.Size([1, 20, 16])


### Chunked-Prefill 实现

chunk 概念与page相似

In [ ]:
x_chunks = x.split(page_size, dim=1)
print(x_chunks)
KVCache = torch.zeros(2, bsz, seq_len, dim)

(tensor([[75, 23, 74,  9, 17, 30, 47, 94]]), tensor([[27, 46, 15, 15, 93, 36, 75, 78]]), tensor([[ 9, 69, 90, 45]]))


编写 chunk-prefill, 但是以下代码有什么逻辑问题？

In [ ]:
def chunk_prefill_method(model, x, page_size, dim, ):

    bsz, seq_len = x.shape
    x_chunks = x.split(page_size, dim=1)
    KVCache = torch.zeros(2, bsz, seq_len, dim)

    num_chunks = len(x_chunks)
    for i, x_c in enumerate(x_chunks):
        bsz, cur_len = x_c.shape
        logits, tmp_KVCache = model(x_c)

        if i == num_chunks-1:  # 最后一个chunk
            last_token_logits = logits[:, -1, :]
            KVCache[0, :, i*page_size: i*page_size+cur_len, :] = tmp_KVCache[0]
            KVCache[1, :, i*page_size: i*page_size+cur_len, :] = tmp_KVCache[1]
        else:
            last_token_logits = None  # 非最后一个 chunk 输出的 logits 是无效的
            KVCache[0, :, i*page_size: (i+1)*page_size, :] = tmp_KVCache[0]
            KVCache[1, :, i*page_size: (i+1)*page_size, :] = tmp_KVCache[1]
    return last_token_logits, KVCache


logits, KVCache = chunk_prefill_method(model, x, page_size, dim)
print(logits.shape)
print(KVCache.shape)

torch.Size([1, 100])
torch.Size([2, 1, 20, 16])


以上的代码问题在于，第 2 个 chunk 计算时，并没有将 第 1 个 chunk-kvcache 输入

```
-    k1, k2, k3, k4
- q1  x,  x, 
- q2  x,  x, 
- q3          x,  x, 
- q4          x,  x, 
```

实际上在 prefill 3,4 时, 将 k1,k2 加载进来


```
-    k1, k2, k3, k4
- q1  x,  x, 
- q2  x,  x, 
- q3  *,  *,  x,  x, 
- q4  *,  *,  x,  x, 
```

此时我们遇到了一种介于 Prefill 和 Decoding 之间的计算模式，区分如下

1. Prefill: 输入 [chunk], kv cache [None]
2. Decoding: 输入 [next_token], kv cache[context]
3. Chunk-prefill: 输入 [chunk], kv cache [chunks]

重写注意力, 实际上写法与 Decoding-Forward 模式相同

In [ ]:
class ChunkedPrefillAttention(Attention):
    def forward(self, X, KVCache):
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)

        if KVCache != None:
            K_cache, V_cache = KVCache[0], KVCache[1]
            K_ = torch.cat([K_cache, K], dim=1)  # cat seq_len dimension
            V_ = torch.cat([V_cache, V], dim=1)  # cat seq_len dimension
        else:
            K_, V_ = K, V

        print('QKV shape', f'{Q.shape}, {K_.shape}, {V_.shape}')
        print('-'*10)

        Z = attention_kernel(Q, K_, V_)
        O = self.Wo(Z)
        return O, [K, V]

In [ ]:
def chunk_prefill_method(model, x, page_size, dim, ):

    bsz, seq_len = x.shape
    x_chunks = x.split(page_size, dim=1)
    KVCache = torch.zeros(2, bsz, seq_len, dim)

    num_chunks = len(x_chunks)
    for i, x_c in enumerate(x_chunks):
        bsz, cur_len = x_c.shape

        # chunk_prefill
        if i == 0:
            chunk_kv_cache = None
        else:
            chunk_kv_cache = KVCache[:, :, :i*page_size]
            print('tmp_kv_cache.shape:', chunk_kv_cache.shape)

        logits, tmp_KVCache = model.forward(x_c, KVCache=chunk_kv_cache)

        if i == num_chunks-1:  # 最后一个chunk
            last_token_logits = logits[:, -1, :]
            KVCache[0, :, i*page_size: i*page_size+cur_len, :] = tmp_KVCache[0]
            KVCache[1, :, i*page_size: i*page_size+cur_len, :] = tmp_KVCache[1]
        else:
            last_token_logits = None  # 非最后一个 chunk 输出的 logits 是无效的
            KVCache[0, :, i*page_size: (i+1)*page_size, :] = tmp_KVCache[0]
            KVCache[1, :, i*page_size: (i+1)*page_size, :] = tmp_KVCache[1]

    return last_token_logits, KVCache

In [ ]:
model = XDGModel(dim,
                 vocab_size,
                 attention_layer=ChunkedPrefillAttention)
x = torch.randint(vocab_size, (bsz, seq_len))
logits, KVCache = chunk_prefill_method(model, x, page_size, dim)
print(logits.shape)
print(KVCache.shape)

QKV shape torch.Size([1, 8, 16]), torch.Size([1, 8, 16]), torch.Size([1, 8, 16])
----------
tmp_kv_cache.shape: torch.Size([2, 1, 8, 16])
QKV shape torch.Size([1, 8, 16]), torch.Size([1, 16, 16]), torch.Size([1, 16, 16])
----------
tmp_kv_cache.shape: torch.Size([2, 1, 16, 16])
QKV shape torch.Size([1, 4, 16]), torch.Size([1, 20, 16]), torch.Size([1, 20, 16])
----------
torch.Size([1, 100])
torch.Size([2, 1, 20, 16])


## Part-3 Mix-PD-Request ChunkPrefill

上述计算逻辑里只处理 Prefill Request, 我们将混合现有的 Decoding Request 进行 step 推理。

一个混合 batch 里可能有以下情况

1. 有 prefill, 有 Decoding
2. 无 prefill, 有 Decoding
3. 有 prefill, 无 Decoding

情况 1 处理情况最复杂。

预处理

1. 获取 Decoding Batch, KVCache
2. 获取 Prefill chunk Batch, 注意a. 在 chunked-prefill 形式下, 也要去拉取 KVCache; b. 最后一个 chunk 需要注意 padding
3. 混合 Batching 注意将输入: Decoding转为 [1, decoding_bsz], Prefill[prefill_bsz, page_size]

计算：

1. attention layer 对不同 P/D batch 做不同处理
2. model 要重写 forward 函数
3. logits 注意取 last-token 位置，而 chunk-prefill 只有最后一个 chunk 的 last-token 对应的 logits 是有效的预测
4. 根据 logits 获取 next-token

更新：

1. 更新解码数据
2. 更新KVCache
3. 更新请求状态

In [ ]:
from collections import deque
from dataclasses import dataclass


@dataclass
class Request:
    """封装单个请求的状态"""
    id: int
    prompt: torch.Tensor = None
    chunked_prompt: List[torch.Tensor] = None
    completions: List[int] = None

    cur_chunk: int = 0
    len_chunk: int = -1
    offset: int = 0

    status: str = 'STATUS_PREFILL'  # STATUS_DECODING, STATUS_COMPLETED
    kv_cache: Optional[torch.Tensor] = None  # kv, 1, seq_len, dim
    max_new_tokens: int = None

In [ ]:
class ChunkedPrefillMergeAttention(Attention):
    def forward(self, X, KVCache_P, KVCache_D, batch_len_p, batch_len_d):
        bsz, seq_len, dim = X.shape

        # proj
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)

        # attention
        start = 0
        if batch_len_d != 0:
            start = 1
            tmp_len = len(batch_len_d)
            Q_d = Q[0, :tmp_len].reshape(tmp_len, 1, dim)
            K_d = K[0, :tmp_len].reshape(tmp_len, 1, dim)
            V_d = V[0, :tmp_len].reshape(tmp_len, 1, dim)
            Z_d = self.forward_pd(Q_d, K_d, V_d, KVCache_D, batch_len_d)
        if batch_len_p != 0:
            Q_p, K_p, V_p = Q[start:], K[start:], V[start:]
            Z_p = self.forward_pd(Q_p, K_p, V_p, KVCache_P, batch_len_p)

        # ouput
        if batch_len_d != 0:
            tmp_len = len(batch_len_d)
            Z_d = Z_d.reshape(1, tmp_len, dim)
            O_d = torch.zeros(1, seq_len, dim)
            O_d[0, :tmp_len] = Z_d
            if batch_len_p != 0:
                O = torch.cat((O_d, Z_p), dim=0)
            else:
                O = O_d
        else:
            O = Z_p

        O = self.Wo(O)

        return O, torch.cat((K.unsqueeze(0), V.unsqueeze(0)), dim=0)

    def forward_pd(self, Q, K, V, KVCache, batch_len):
        bsz, seq_len, dim = Q.shape
        nkv, bsz, kv_len, dim = KVCache.shape

        # print('seq_len:', seq_len)
        # print('kv_len:', kv_len)

        new_kv_len = seq_len + kv_len

        new_KVCache = torch.zeros(nkv, bsz, new_kv_len, dim)

        for i, l in enumerate(batch_len):
            # 历史 kV
            if l != 0:
                new_KVCache[:, i, :l] = KVCache[:, i, :l]

            # 当前 KV
            new_KVCache[0, i, l:l+seq_len] = K[i, :, :]
            new_KVCache[1, i, l:l+seq_len] = V[i, :, :]

        Z = attention_kernel(Q, new_KVCache[0], new_KVCache[1])
        return Z


class ChunkedPrefillTransformer(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.embd = nn.Embedding(vocab_size, dim)
        self.attn = ChunkedPrefillMergeAttention(dim)
        self.lm_head = nn.Linear(dim, vocab_size)

    def forward(self, input_pd, KVCache_P, KVCache_D, batch_len_p, batch_len_d):
        X = self.embd(input_pd)
        H, KV = self.attn(X, KVCache_P, KVCache_D, batch_len_p, batch_len_d)
        logits = self.lm_head(H)
        return logits, KV


model = ChunkedPrefillTransformer(vocab_size, dim)

In [ ]:
import random


class Scheduler:
    def __init__(self, page_size: int = -1,
                 num_requests: int = -1,
                 vocab_size: int = 100,
                 min_prompt_len=32,
                 max_prompt_len=128,
                 max_new_tokens=32,):
        self.vocab_size = vocab_size
        self.page_size = page_size

        self.min_prompt_len = min_prompt_len
        self.max_prompt_len = max_prompt_len
        self.max_new_tokens = max_new_tokens

        self.reqs_p = set()
        self.reqs_d = set()
        self.reqs_c = set()  # completed
        self.max_num_requests = num_requests
        self.reqs = []
        if num_requests != -1:
            self.dummy_reqeusts(num_requests)

    def dummy_reqeusts(self, N):
        for i in range(N):
            prompt_len = random.randint(
                self.min_prompt_len, self.max_prompt_len)
            prompt = torch.randint(self.vocab_size, size=(1, prompt_len))[0]
            chunked_prompt = prompt.split(page_size, dim=0)
            req = Request(id=i,
                          kv_cache=None,
                          prompt=prompt,
                          chunked_prompt=chunked_prompt,
                          max_new_tokens=random.randint(
                              10, self.max_new_tokens),
                          )
            self.reqs.append(req)
            self.reqs_p.add(i)

    def get_prefill_batch(self, ):
        """"""
        N = len(self.reqs_p)
        if N == 0:
            return {
                'input_ids': None,
                'kv_cache': None,
                'batch_to_request': None,
                'len': 0,
            }
        else:
            # 处理批量输入
            batch_input_ids = torch.zeros(N, page_size, dtype=torch.long)
            kv_lens = []
            batch_len = []
            batch_to_request = {}
            for B_id, R_id in enumerate(self.reqs_p):
                req = self.reqs[R_id]
                chunk = req.chunked_prompt[req.cur_chunk]
                kv_cache = req.kv_cache

                kv_lens.append(req.offset)  # offset is kvlen

                batch_to_request[B_id] = R_id

                chunk_len = len(chunk)
                batch_len.append(req.offset)
                batch_input_ids[B_id, :chunk_len] = torch.tensor(
                    chunk, dtype=torch.long).clone().detach()

            # 处理批量 KV Cache
            max_kv_lens = max(kv_lens)
            batch_kv_cache = torch.zeros(2, N, max_kv_lens, dim)
            for i in range(N):
                req = self.reqs[i]
                if req.kv_cache == None:
                    continue
                else:
                    batch_kv_cache[:, i, :req.offset] = req.kv_cache

            return {
                'input_ids': batch_input_ids,
                'kv_cache': batch_kv_cache,
                'batch_to_request': batch_to_request,
                'len': batch_len,
            }

    def get_decoding_batch(self, ):
        """"""

        N = len(self.reqs_d)
        if N == 0:
            return {
                'input_ids': None,
                'kv_cache': None,
                'batch_to_request': None,
                'len': 0,
            }
        else:
            batch_len = []
            batch_to_request = {}
            # 填充 pad token
            # 当成 bsz=1， seq_len=decoding_request_num
            batch_input_ids = torch.zeros(1, self.page_size, dtype=torch.long)

            for B_id, R_id in enumerate(self.reqs_d):
                req = self.reqs[R_id]
                batch_input_ids[0, B_id] = req.completions[-1]
                batch_len.append(len(req.prompt) + len(req.completions))
                batch_to_request[B_id] = R_id

            max_kv_len = max(batch_len)
            batch_kv_cache = torch.zeros(2, N, max_kv_len, dim)
            for i in range(N):
                R_id = batch_to_request[B_id]
                req = self.reqs[R_id]
                if req.kv_cache == None:
                    continue
                else:
                    batch_kv_cache[:, i, :req.offset] = req.kv_cache

            return {
                'input_ids': batch_input_ids,
                'kv_cache': batch_kv_cache,
                'batch_to_request': batch_to_request,
                'len': batch_len,
            }

    def get_merge_batch(self,):
        prefill_batch = self.get_prefill_batch()
        decoding_batch = self.get_decoding_batch()

        if len(self.reqs_p) != 0 and len(self.reqs_d) != 0:
            input_ids = torch.cat(
                (decoding_batch['input_ids'], prefill_batch['input_ids']), dim=0)
        elif len(self.reqs_p) != 0 and len(self.reqs_d) == 0:
            input_ids = prefill_batch['input_ids']
        elif len(self.reqs_p) == 0 and len(self.reqs_d) != 0:
            input_ids = decoding_batch['input_ids']
        else:
            input_ids = None

        return input_ids, prefill_batch, decoding_batch

    def update_request(self,
                       next_token,
                       batch_to_request_p,
                       batch_to_request_d,
                       kv_p,
                       kv_d):

        # decoding request update
        start = 0
        if len(self.reqs_d) > 0:
            start = 1
            for B_id, R_id in batch_to_request_d.items():
                req = self.reqs[R_id]
                token = next_token[0, B_id]
                if self.reqs[R_id].completions is None:
                    self.reqs[R_id].completions = [token]
                else:
                    self.reqs[R_id].completions.append(token)

                self.reqs[R_id].kv_cache = torch.cat(
                    (req.kv_cache, kv_d[:, B_id, None]), dim=1)  # seq_len dim cat
                self.reqs[R_id].offset += 1

                # max_new_token = 20
                if self.reqs[R_id].offset == len(req.prompt) + req.max_new_tokens:
                    self.reqs[R_id].status = "STATUS_COMPLETED"
                    self.reqs_d.remove(R_id)
                    self.reqs_c.add(R_id)
                    l = len(self.reqs[R_id].completions)
                    print(f'finish.ID:{R_id}, completions len{l}')

        # prefill request update
        if len(self.reqs_p) > 0:
            next_token = next_token[start:]

            for B_id, R_id in batch_to_request_p.items():
                req = self.reqs[R_id]

                valid_len = len(req.chunked_prompt[req.cur_chunk])
                self.reqs[R_id].offset += valid_len

                if req.cur_chunk == len(req.chunked_prompt)-1:

                    token = next_token[B_id, valid_len-1]
                    if self.reqs[R_id].completions is None:
                        self.reqs[R_id].completions = [token]
                    else:
                        self.reqs[R_id].completions.append(token)

                    self.reqs[R_id].status = "STATUS_DECODING"
                    self.reqs_p.remove(R_id)
                    self.reqs_d.add(R_id)

                if req.kv_cache is not None:

                    self.reqs[R_id].kv_cache = torch.cat((self.reqs[R_id].kv_cache,
                                                          kv_p[:, B_id, :valid_len]), dim=1)
                else:
                    self.reqs[R_id].kv_cache = kv_p[:, B_id, :valid_len]

                self.reqs[R_id].cur_chunk += 1

    def is_finish(self, ):
        return len(self.reqs_c) == self.max_num_requests

    def get_info(self,):
        # 'STATUS_PREFILL # STATUS_DECODING # STATUS_COMPLETED
        info_table = []
        for req in self.reqs:
            if req.status == 'STATUS_PREFILL':
                info_table.append('0')
            elif req.status == 'STATUS_DECODING':
                info_table.append('*')
            else:
                info_table.append('-')
        return ''.join(info_table)


scheduler = Scheduler(vocab_size=vocab_size,
                      page_size=page_size, num_requests=6)
print(scheduler.reqs_p)

{0, 1, 2, 3, 4, 5}


In [ ]:
class ChunkPrefillEngine:
    def __init__(self, scheduler, model):
        self.scheduler = scheduler
        self.model = model

    def step(self, ):
        merge_input_ids, prefill_batch, decoding_batch = self.scheduler.get_merge_batch()

        logits, KV = self.model(merge_input_ids,
                                prefill_batch['kv_cache'],
                                decoding_batch['kv_cache'],
                                prefill_batch['len'],
                                decoding_batch['len'],)

        next_token = torch.argmax(logits, dim=-1)
        KV_P = None
        KV_D = None
        if len(self.scheduler.reqs_d) != 0:
            KV_D = KV[:, 0, :]
            KV_P = KV[:, 1:, :]
        else:
            KV_P = KV[:, 0:, :]

        self.scheduler.update_request(next_token,
                                      prefill_batch['batch_to_request'],
                                      decoding_batch['batch_to_request'],
                                      KV_P,
                                      KV_D,)

In [ ]:
page_size = 16
model = ChunkedPrefillTransformer(vocab_size, dim)
scheduler = Scheduler(vocab_size=vocab_size,
                      page_size=page_size,
                      num_requests=page_size-1,
                      max_prompt_len=512,
                      max_new_tokens=20)
engine = ChunkPrefillEngine(scheduler, model)
n_step = 0
while 1:
    n_step += 1

    with torch.no_grad():
        engine.step()

    info = engine.scheduler.get_info()
    print('step:', n_step, '\t\t\tinfo', info)

    if engine.scheduler.is_finish():
        break
    # break

step: 1 			info 000000000000000
step: 2 			info 000000000000000
step: 3 			info 000000000000000
step: 4 			info 000000000000000
step: 5 			info 000000000000000
step: 6 			info 000000000000000
step: 7 			info 0*0000*00000000
step: 8 			info 0*0000*00000000
step: 9 			info 0*0000*00000000
step: 10 			info 0*0000*00000000
step: 11 			info 0*0000*00000000
step: 12 			info 0*0000*00000000
step: 13 			info 0*0000*000*0000
step: 14 			info 0*0000*000*0000
step: 15 			info 0*000**000**00*
step: 16 			info 0*000**000**00*
step: 17 			info 0**00**000**00*
step: 18 			info 0**00**000**00*
step: 19 			info 0**00**000**00*
step: 20 			info 0**00**000**00*
step: 21 			info 0**00**000**00*
step: 22 			info 0**00**000**0**
step: 23 			info 0**00**000*****
finish.ID:1, completions len18
step: 24 			info 0-*00***00*****
finish.ID:6, completions len19
step: 25 			info *-*00*-*00*****
step: 26 			info *-*00*-*00*****
finish.ID:10, completions len15
step: 27 			info *-*0**-**0-****
finish.ID:11, completion

/var/folders/zg/dkd7345140x57tz11tnwcrdm0000gn/T/ipykernel_61012/3196038793.py:66: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  batch_input_ids[B_id, :chunk_len] = torch.tensor(chunk, dtype=torch.long).clone().detach()


## 分析

### Chunked Prefill 对推理系统设计影响

1. 找到各模块共性计算，进行融合 PD
2. 上述例子描述了 proj 类算子，我们需要进一步分析，注意力计算是否有类似的PD计算共性。

所幸 PD 的注意力遵循：

1. Decoding `单 q 多 KV`
2. Prefill 虽然注意力是 `多 q 多 KV` 计算的， 但其子任务仍为 `单 q 多 KV`

为了节省存储开销，原本所遵循的 kernel 区分了

```
forward_prefill(), page_attention_prefill_kernel()
forward_decoding(), page_attention_decoding_kernel()
```

目标要实现一种不区分 PD 的通用 kernel

```
forward_chunk_prefill(), page_attention_kernel()
```

这一块并没有更近一步的实现。

### Chunked Prefill 调度逻辑优化

定义 prompt_len 短的请求集合记为 `Req_s`, 长请求集合记为 `Req_l`

1. 假设一批长度不等的请求中，应当将 `Req_s` 提前调度，能够快速结束 prefill 进入到 decoding
2. 对 `Req_l` 请求逐个 step 进行 chunked-prefill, 此过程将第一步的 `Req_s` 的 decoding 计算任务带上
3. 最后进入到全 decoding 阶段

### 当前版本实现缺陷

1. chunk-prefill 过程得到的 chunk-kv-cache 可以由 page-kv-cache 进行管理
2. chunk-prefill 并未如期望的实现了 PD 混合的 attention 计算。但其他模块由于 merge batch 实际上完美做到了搭便车

### chunked-prefill 优化思考

1. 在 merge-batch 中，如何合理分配 P/D batch 数量？ 
2. merge-batch、分离output、更新请求/KVcache 实际上操作比想象中多得多，如何设计合理的数据结构来简化操作（而不只是方便写代码那么简单）
3. chunked-prefill 是在线多轮对话场景中最匹配的场景，天然能做 prefix-caching

### chunked-prefill 与 vLLM 区别

1. chunked-prefill 分可能多个 step 完成，而 vLLM Prefill 一步完成
2. vLLM 行为是 request-level 实现 kernel 计算。 chunked-prefill 在本 notebook 实现中是 batch 级别的。

chunked-prefill 集成在 vLLM-V1 版本中，将在下一个notebook讲解 V1 的实现。